In [1]:
import os
import gc
import pandas as pd
import numpy as np
import tensorflow as tf
from src.common.utils import get_root_directory,collate_array_elements, gridsearch
import matplotlib.pyplot as plt
from src.preprocessing.data_loader import DataLoader
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, accuracy_score, precision_score
from sklearn.preprocessing import MinMaxScaler
from src.common.stats import root_mean_squared_percentage_error
import mlflow
import mlflow.keras
import keras
import timeshap
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, InputLayer, Reshape, Bidirectional, GRU
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError, MeanSquaredError
tf.keras.mixed_precision.set_global_policy('mixed_float16')

2025-08-16 16:35:17.950148: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-16 16:35:18.527159: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755362118.733844    2871 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755362118.796971    2871 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-16 16:35:19.340131: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPUs available: {[gpu.name for gpu in gpus]}")
else:
    print("No GPU available.")

GPUs available: ['/physical_device:GPU:0']


In [3]:
#PARAMETERS TO CHANGE
model_type = "BiGRU"

n_periods = 7
start_date = "01/01/2021"
end_date = "31/12/2023"
split_size = 0.15

selected_data = []
sentiment_type = "cryptobert"
ETH_base = ['ETH_D_AvgPrc']
ETH_blk = False
ETH_features = ['ETH_D_PrcDir']
BTC_features = ['BTC_D_AvgPrc']
LTC_features = ['LTC_D_AvgPrc']
sentiment_features = ['D_VADER_AvgScr_Ex', 'D_VADER_Sent_AvgEx','D_FINBERT_AvgScr_Ex','D_FINBERT_Sent_AvgEx','D_CRYPTOBERT_AvgScr_Ex','D_CRYPTOBERT_Sent_AvgEx']
filtered_features = ETH_base
run_name = "ETH_base"
if ETH_blk:
    filtered_features.extend(ETH_features)
    run_name = run_name +"_ETH_blk"
if 'BTC' in selected_data:
    filtered_features = filtered_features + BTC_features
    run_name = run_name +"_BTC_blk"
if 'LTC' in selected_data:
    filtered_features = filtered_features + LTC_features
    run_name = run_name +"_LTC_blk"
if 'sentiment_analysis' in selected_data:
    filtered_features = filtered_features + [feature for feature in sentiment_features if sentiment_type.upper() in feature]
    run_name = run_name +f"_sent_{sentiment_type}"

split_name = "test"
experiment_name = f"{model_type}_train_{split_name}"

In [4]:
param_grid = {
    "lags": [14],
    "layers":[3],
    "dropout":[0.05],
    "units":[32],
    "batch_size":[32],
    "learning_rate":[0.01],
    "epochs":[37],
    "patience":[20]
}

In [5]:
@gridsearch(param_grid)
def run_experiment(lags, layers, dropout, units, batch_size,learning_rate,epochs,patience):
    
    root_dir = get_root_directory()
    DL = DataLoader(root_dir)
    DL.load_data()
    DL.merge_selected_data(selected_data=selected_data)
    DL.select_features(features=filtered_features)
    DL.set_time_range(start_date=start_date, end_date=end_date)
    random_state=42
    np.random.seed(random_state)
    tf.random.set_seed(random_state)
    
    if split_name=="test":
        train, test = DL.split_data(split_size=split_size, overlap=True, overlap_size=lags)
        split = test
    elif split_name=="validation":
        train, test = DL.split_data(split_size=split_size)
        train, val = DL.split_data(split_type="train_val", split_size=split_size, overlap=True, overlap_size=lags)
        split = val
    y_train, x_train = DL.generate_X_y_tensors(train, data_split="train", lags=lags, horizon=n_periods)
    y_val, x_val = DL.generate_X_y_tensors(split, data_split=split_name, lags=lags, horizon=n_periods)

    model=Sequential()
    model.add(InputLayer((lags,len(filtered_features))))
    if layers>1:
        for i in range(layers):
            model.add(Bidirectional(GRU(units=units, return_sequences=True)))
            if dropout>0:
                model.add(Dropout(rate=dropout))
    model.add(Bidirectional(GRU(units=units, return_sequences=False)))
    if dropout>0:
        model.add(Dropout(rate=dropout))
    model.add(Dense(n_periods))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse', metrics=[RootMeanSquaredError()])
    early_stop = EarlyStopping(monitor='val_loss', restore_best_weights=True, patience=patience, verbose=0)
    
    mlflow.set_tracking_uri(f"sqlite:///{root_dir}/mlruns/{model_type}/mlruns2.db")
    mlflow.set_experiment(experiment_name)
    with mlflow.start_run(run_name=run_name):
        if split_name=="validation":
            model_history = model.fit(x=x_train, y=y_train, validation_data=(x_val,y_val), epochs=epochs, batch_size=batch_size, callbacks=[early_stop],verbose=0)
            best_epoch = early_stop.best_epoch + 1 
            epoch_record = best_epoch
            y_pred = model.predict(x=x_val)
        else:
            model_history = model.fit(x=x_train, y=y_train, epochs=epochs, batch_size=batch_size,verbose=0)
            y_pred = model.predict(x=x_val)
            epoch_record = epochs
            model_path = str(os.path.join(root_dir,"mlruns",model_type,f"{model_type}_best"))
            model.export(model_path)
        mlflow.log_params({
            "lags":lags,
            "layers": layers,
            "dropout": dropout,
            "units": units,
            "batch_size":batch_size,
            "learning_rate":learning_rate,
            "epoch": epoch_record,
            "n_periods": n_periods,
            "prediction_start_date":pd.to_datetime(y_val[0,:].Date.values).strftime('%d-%m-%Y'),
            "features":filtered_features
        })
        y_scaler, _ = DL.get_scalers()
        df = pd.DataFrame(y_scaler.inverse_transform(y_pred), columns=[f"t{i+1}" for i in range(n_periods)])
        y_val_unscaled = y_scaler.inverse_transform(y_val)
        mse_list = []
        rmse_list = []
        rmspe_list = []
        mae_list = []
        mape_list = []
        accuracy_list  = []
        precision_list = []
        for i in range(n_periods):
            df[f't{i+1}_prc_dir'] = df[f't{i+1}'].diff().apply(lambda x: 1 if x > 0 else -1)
            mse = mean_squared_error(y_val_unscaled[:,i], df[f't{i+1}'])
            rmse = root_mean_squared_error(y_val_unscaled[:,i],  df[f't{i+1}'])
            rmspe = root_mean_squared_percentage_error(y_val_unscaled[:,i],  df[f't{i+1}'])
            mae = mean_absolute_error(y_val_unscaled[:,i],  df[f't{i+1}'])
            mape = mean_absolute_percentage_error(y_val_unscaled[:,i],  df[f't{i+1}'])
            accuracy = accuracy_score(pd.Series(y_val_unscaled[:,i]).diff().apply(lambda x: 1 if x > 0 else -1).dropna(), df[f't{i+1}_prc_dir'].dropna())
            precision = precision_score(pd.Series(y_val_unscaled[:,i]).diff().apply(lambda x: 1 if x > 0 else -1).dropna(), df[f't{i+1}_prc_dir'].dropna())
            mse_list.append(mse)
            rmse_list.append(rmse)
            rmspe_list.append(rmspe)
            mae_list.append(mae)
            mape_list.append(mape)
            accuracy_list.append(accuracy)
            precision_list.append(precision)
            avg_mse = sum(mse_list)/len(mse_list)
            avg_rmspe = sum(rmspe_list)/len(rmspe_list)
            avg_rmse = sum(rmse_list)/len(rmse_list)
            avg_mae = sum(mae_list)/len(mae_list)
            avg_mape = sum(mape_list)/len(mape_list)
            avg_accuracy = sum(accuracy_list)/len(accuracy_list)
            avg_precision = sum(precision_list)/len(precision_list)
            mlflow.log_metrics({
                        "mse_daily":mse_list[i], 
                        "rmse_daily":rmse_list[i],
                        "rmspe_daily":rmspe_list[i], 
                        "mae_daily":mae_list[i], 
                        "mape_daily":mape_list[i], 
                        "accuracy_daily":accuracy_list[i],
                        "precision_daily":precision_list[i],
                        "avg_mse": avg_mse,
                        "avg_rmse": avg_rmse,
                        "avg_rmspe": avg_rmspe,
                        "avg_mae": avg_mae,
                        "avg_mape": avg_mape,
                        "avg_accuracy":avg_accuracy,
                        "avg_precision":avg_precision
                    }, step=(i+1))
    mlflow.end_run()
    if split_name=="test":
        results_file = f"{run_name}_{split_name}.csv"
        path = str(os.path.join(root_dir,"mlruns",model_type))
        df.to_csv(str(os.path.join(path,results_file)), index=False)
    gc.collect()

In [6]:
run_experiment()

Running with params: {'lags': 14, 'layers': 3, 'dropout': 0.05, 'units': 32, 'batch_size': 32, 'learning_rate': 0.01, 'epochs': 37, 'patience': 20}


I0000 00:00:1755362167.072715    2871 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13760 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:1e.0, compute capability: 7.5
2025-08-16 16:36:14.431695: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
I0000 00:00:1755362174.538959    2997 cuda_dnn.cc:529] Loaded cuDNN version 90501


5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 123ms/step
INFO:tensorflow:Assets written to: /home/ec2-user/ETH_Price_Predition/mlruns/BiGRU/BiGRU_best/assets


INFO:tensorflow:Assets written to: /home/ec2-user/ETH_Price_Predition/mlruns/BiGRU/BiGRU_best/assets


Saved artifact at '/home/ec2-user/ETH_Price_Predition/mlruns/BiGRU/BiGRU_best'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 14, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float16, name=None)
Captures:
  140315828422416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140315828424912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140315828425680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140315828424528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140315828426256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140315828426448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140315828428368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140315828429328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140315828430096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140315828429136: TensorSpec(shape=(), dtype=tf.r